# 05 — Data Modeling: подготовка snapshot-датасета

Ноутбук строит табличный датасет для двухстадийной Hurdle-модели: классификатор оценивает вероятность положительного GMV, а регрессор — сумму GMV при положительном исходе.

**Вход:** `data/train.parquet`, события с 2025-01-01 по 2026-02-13.  
**Выход:** `data/Prepared_data.parquet`.  
**Grain:** одна строка = `user_id × cutoff_date`.


## Временная схема и назначение snapshot

Признаки используют только события `event_date <= cutoff_date`. Для первых десяти cutoff target рассчитывается по событиям `cutoff_date < event_date <= cutoff_date + 30 дней`. Пользователи, впервые появившиеся после cutoff, в соответствующий snapshot не входят.

Размеченные cutoff: `2025-04-19`, `2025-05-19`, `2025-06-18`, `2025-07-18`, `2025-08-17`, `2025-09-16`, `2025-10-16`, `2025-11-15`, `2025-12-15`, `2026-01-14`.

Inference-cutoff: `2026-02-13`. Его признаки используют всю доступную историю, а обе target-колонки равны `null`; строки предназначены для прогноза GMV за 2026-02-14—2026-03-15.


## Архитектура расчёта

`train.parquet` читается PyArrow по row group. Так как исходные данные отсортированы по `user_id, event_date`, незавершённая история последнего пользователя переносится в следующий блок, после чего все 11 snapshot пользователя вычисляются одним проходом через NumPy. Выход буферизуется по cutoff, записывается во временные Parquet-файлы и объединяется в один файл, отсортированный по `cutoff_date, user_id`.

В памяти одновременно находятся один row group, история одного пользователя и небольшие выходные буферы. Финальный файл заменяется только после успешной полной проверки; временные файлы удаляются.


## Полный словарь итоговых колонок

Таблица ниже является контрактом порядка, типов, формул и семантики всех 95 физических колонок: 2 служебные, 91 модельный признак и 2 target. Ноль означает определённое отсутствие события; `NaN/null` означает математически или поведенчески неопределённое значение.

| Группа | Колонка | Тип | Формула | Окно | Интерпретация | Пропуски | Назначение |
|---|---|---|---|---|---|---|---|
| Служебные | `user_id` | `int64` | Идентификатор пользователя из train.parquet | — | Ключ пользователя; в модель как обычный признак не передаётся | Не допускается | Служебная |
| Служебные | `cutoff_date` | `date32` | Дата, относительно которой разделяются история и будущее | — | Временной ключ snapshot и основа out-of-time split | Не допускается | Служебная |
| Lifetime-агрегаты | `searches_total` | `int32` | sum(searches) | Вся история до cutoff включительно | Lifetime-объём поисковых запросов | 0 означает отсутствие событий | Classifier + regressor |
| Lifetime-агрегаты | `cart_items_total` | `int32` | sum(to_cart) | Вся история до cutoff включительно | Lifetime-число добавленных в корзину товаров | 0 означает отсутствие событий | Classifier + regressor |
| Lifetime-агрегаты | `purchased_items_total` | `int32` | sum(to_ord) | Вся история до cutoff включительно | Lifetime-число купленных товаров | 0 означает отсутствие событий | Classifier + regressor |
| Lifetime-агрегаты | `search_purchased_items_total` | `int32` | sum(search_to_ord) | Вся история до cutoff включительно | Lifetime-число товаров, купленных через поиск | 0 означает отсутствие событий | Classifier + regressor |
| Lifetime-агрегаты | `gmv_total` | `float32` | sum(gmv) | Вся история до cutoff включительно | Накопленный исторический GMV | 0 означает отсутствие GMV | Classifier + regressor |
| Lifetime-агрегаты | `gmv_search_total` | `float32` | sum(gmv_search) | Вся история до cutoff включительно | Накопленный GMV поискового канала | 0 означает отсутствие GMV | Classifier + regressor |
| Lifetime-агрегаты | `gmv_cat_total` | `float32` | sum(gmv_cat) | Вся история до cutoff включительно | Накопленный GMV каталожного канала | 0 означает отсутствие GMV | Classifier + regressor |
| Lifetime-агрегаты | `purchase_days` | `int32` | Число дат, в которые to_ord > 0 | Вся история до cutoff включительно | Частота покупочных дней, а не число заказов | 0 означает отсутствие покупок | Classifier + regressor |
| RFM | `days_since_last_purchase` | `float32` | cutoff_date − max(event_date при to_ord > 0) | Lifetime | Давность последней покупки; больше означает более долгую паузу | NaN, если покупок не было | Преимущественно classifier |
| RFM | `never_purchased` | `int8` | 1, если до cutoff нет to_ord > 0 | Lifetime | Явно отделяет отсутствие покупок от недавней покупки | Не допускается | Преимущественно classifier |
| RFM | `days_since_last_search` | `float32` | cutoff_date − max(event_date при searches > 0) | Lifetime | Давность последнего поискового интереса | NaN, если поисков не было | Преимущественно classifier |
| RFM | `never_searched` | `int8` | 1, если до cutoff нет searches > 0 | Lifetime | Явно отмечает отсутствие поисковой истории | Не допускается | Преимущественно classifier |
| RFM | `recency_ratio` | `float32` | days_since_last_purchase / median_purchase_gap_days | Lifetime | Пауза относительно типичного персонального интервала; выше 1 означает запаздывание | NaN без покупки, второго покупочного дня или при нулевом интервале | Преимущественно classifier |
| RFM | `active_days` | `int32` | Число наблюдаемых дат пользователя | Lifetime | Общая регулярность активности | Не допускается | Classifier + regressor |
| RFM | `purchase_frequency` | `float32` | purchase_days / active_days | Lifetime | Доля активных дней с покупкой | NaN только при невозможном нулевом active_days | Classifier + regressor |
| RFM | `search_to_purchase_freq` | `float32` | searches_total / purchased_items_total | Lifetime | Количество поисковых запросов на один купленный товар | NaN, если покупок не было | Classifier + regressor |
| RFM | `gmv_per_item` | `float32` | gmv_total / purchased_items_total | Lifetime | Средний GMV на купленный товар; не средний чек заказа | NaN, если покупок не было | Преимущественно regressor |
| RFM | `gmv_daily_mean` | `float32` | gmv_total / customer_age_days | Lifetime | Средний GMV на календарный день жизни пользователя | Не допускается | Преимущественно regressor |
| Микроворонки | `sci_items_per_query` | `float32` | search_purchased_items_total / searches_total | Lifetime | Товаров, купленных через поиск, на один запрос | NaN, если поисков не было | Classifier + regressor |
| Микроворонки | `cart_abandonment_rate` | `float32` | clip(1 − purchased_items_total / cart_items_total, 0, 1) | Lifetime | Прокси доли добавленных, но не купленных товаров | NaN, если добавлений не было | Преимущественно classifier |
| Микроворонки | `cart_purchase_exceeds_cart` | `int8` | 1, если purchased_items_total / cart_items_total > 1 | Lifetime | Отмечает несогласованность доступной истории корзины и покупок | 0 при отсутствии корзины | Classifier + regressor |
| Микроворонки | `search_gmv_share` | `float32` | gmv_search_total / gmv_total | Lifetime | Доля исторического GMV поискового канала | NaN при нулевом GMV | Classifier + regressor |
| Объёмы по окнам | `searches_7d` | `int32` | sum(searches) | cutoff−6 … cutoff | Объём поисковых запросов за последние 7 дней | 0 означает отсутствие событий | Classifier + regressor |
| Объёмы по окнам | `searches_30d` | `int32` | sum(searches) | cutoff−29 … cutoff | Объём поисковых запросов за последние 30 дней | 0 означает отсутствие событий | Classifier + regressor |
| Объёмы по окнам | `searches_prev_30d` | `int32` | sum(searches) | cutoff−59 … cutoff−30 | Объём поисковых запросов за предыдущие 30 дней | 0 означает отсутствие событий | Classifier + regressor |
| Объёмы по окнам | `searches_90d` | `int32` | sum(searches) | cutoff−89 … cutoff | Объём поисковых запросов за последние 90 дней | 0 означает отсутствие событий | Classifier + regressor |
| Объёмы по окнам | `cart_items_7d` | `int32` | sum(to_cart) | cutoff−6 … cutoff | Объём добавленных товаров за последние 7 дней | 0 означает отсутствие событий | Classifier + regressor |
| Объёмы по окнам | `cart_items_30d` | `int32` | sum(to_cart) | cutoff−29 … cutoff | Объём добавленных товаров за последние 30 дней | 0 означает отсутствие событий | Classifier + regressor |
| Объёмы по окнам | `cart_items_prev_30d` | `int32` | sum(to_cart) | cutoff−59 … cutoff−30 | Объём добавленных товаров за предыдущие 30 дней | 0 означает отсутствие событий | Classifier + regressor |
| Объёмы по окнам | `cart_items_90d` | `int32` | sum(to_cart) | cutoff−89 … cutoff | Объём добавленных товаров за последние 90 дней | 0 означает отсутствие событий | Classifier + regressor |
| Объёмы по окнам | `purchased_items_7d` | `int32` | sum(to_ord) | cutoff−6 … cutoff | Объём купленных товаров за последние 7 дней | 0 означает отсутствие событий | Classifier + regressor |
| Объёмы по окнам | `purchased_items_30d` | `int32` | sum(to_ord) | cutoff−29 … cutoff | Объём купленных товаров за последние 30 дней | 0 означает отсутствие событий | Classifier + regressor |
| Объёмы по окнам | `purchased_items_prev_30d` | `int32` | sum(to_ord) | cutoff−59 … cutoff−30 | Объём купленных товаров за предыдущие 30 дней | 0 означает отсутствие событий | Classifier + regressor |
| Объёмы по окнам | `purchased_items_90d` | `int32` | sum(to_ord) | cutoff−89 … cutoff | Объём купленных товаров за последние 90 дней | 0 означает отсутствие событий | Classifier + regressor |
| Объёмы по окнам | `gmv_7d` | `float32` | sum(gmv) | cutoff−6 … cutoff | Объём GMV за последние 7 дней | 0 означает отсутствие событий | Classifier + regressor |
| Объёмы по окнам | `gmv_30d` | `float32` | sum(gmv) | cutoff−29 … cutoff | Объём GMV за последние 30 дней | 0 означает отсутствие событий | Classifier + regressor |
| Объёмы по окнам | `gmv_prev_30d` | `float32` | sum(gmv) | cutoff−59 … cutoff−30 | Объём GMV за предыдущие 30 дней | 0 означает отсутствие событий | Classifier + regressor |
| Объёмы по окнам | `gmv_90d` | `float32` | sum(gmv) | cutoff−89 … cutoff | Объём GMV за последние 90 дней | 0 означает отсутствие событий | Classifier + regressor |
| Velocity | `search_velocity_7d_vs_30d` | `float32` | (searches_7d / exposure_7d) / (searches_30d / exposure_30d) | 7d против 30d | Краткосрочное ускорение активности относительно месяца | NaN при нулевой месячной базе | Преимущественно classifier |
| Velocity | `search_velocity_30d_vs_lifetime` | `float32` | (searches_30d / exposure_30d) / (searches_total / customer_age_days) | 30d против lifetime | Месячная интенсивность относительно персональной lifetime-нормы | NaN при нулевой lifetime-базе | Classifier + regressor |
| Velocity | `cart_velocity_7d_vs_30d` | `float32` | (cart_items_7d / exposure_7d) / (cart_items_30d / exposure_30d) | 7d против 30d | Краткосрочное ускорение активности относительно месяца | NaN при нулевой месячной базе | Преимущественно classifier |
| Velocity | `cart_velocity_30d_vs_lifetime` | `float32` | (cart_items_30d / exposure_30d) / (cart_items_total / customer_age_days) | 30d против lifetime | Месячная интенсивность относительно персональной lifetime-нормы | NaN при нулевой lifetime-базе | Classifier + regressor |
| Velocity | `purchase_velocity_7d_vs_30d` | `float32` | (purchased_items_7d / exposure_7d) / (purchased_items_30d / exposure_30d) | 7d против 30d | Краткосрочное ускорение активности относительно месяца | NaN при нулевой месячной базе | Преимущественно classifier |
| Velocity | `purchase_velocity_30d_vs_lifetime` | `float32` | (purchased_items_30d / exposure_30d) / (purchased_items_total / customer_age_days) | 30d против lifetime | Месячная интенсивность относительно персональной lifetime-нормы | NaN при нулевой lifetime-базе | Classifier + regressor |
| Тренды | `intent_trend_log` | `float32` | log1p(cart_items_30d) − log1p(cart_items_prev_30d) | 30d против prev30d | Рост или снижение намерения, выраженного корзиной | Не допускается | Преимущественно classifier |
| Стабильность | `engagement_daily_mean_90d` | `float32` | sum(searches + to_cart + to_ord) / exposure_90d | Последние доступные 90 дней | Средняя календарно-дневная вовлечённость с учётом нулевых дней | Не допускается | Classifier + regressor |
| Стабильность | `engagement_cv_90d` | `float32` | std_daily(searches + to_cart + to_ord) / engagement_daily_mean_90d | Последние доступные 90 дней | Всплесковость поведения; больше означает меньшую регулярность | NaN при нулевой средней вовлечённости | Преимущественно classifier |
| Стабильность | `no_recent_engagement` | `int8` | 1, если сумма searches + to_cart + to_ord за доступные 90 дней равна 0 | Последние доступные 90 дней | Отсутствие недавней воронки взаимодействия | Не допускается | Преимущественно classifier |
| Возраст | `customer_age_days` | `int32` | cutoff_date − first_activity_date + 1 | Lifetime | Наблюдаемый возраст пользователя в календарных днях | Не допускается | Classifier + regressor |
| Возраст | `customer_age_left_censored` | `int8` | 1, если first_activity_date = 2025-01-01 | Lifetime | Реальный возраст может начинаться до доступного датасета | Не допускается | Classifier + regressor |
| Сезонность | `last_visit_weekday_sin` | `float32` | sin(2π × weekday(last_activity) / 7) | Последнее событие до cutoff | Циклическое кодирование дня недели последней активности | Не допускается | Преимущественно classifier |
| Сезонность | `last_visit_weekday_cos` | `float32` | cos(2π × weekday(last_activity) / 7) | Последнее событие до cutoff | Вторая координата дня недели последней активности | Не допускается | Преимущественно classifier |
| Сезонность | `cutoff_month_sin` | `float32` | sin(2π × (month(cutoff) − 1) / 12) | Cutoff | Циклическая сезонность месяца прогноза | Не допускается | Classifier + regressor |
| Сезонность | `cutoff_month_cos` | `float32` | cos(2π × (month(cutoff) − 1) / 12) | Cutoff | Вторая координата сезонности месяца | Не допускается | Classifier + regressor |
| Whale | `whale_score` | `float32` | percentile_rank(gmv_90d) внутри одного cutoff | Snapshot | Относительное положение пользователя по недавнему GMV | Не допускается | Преимущественно regressor |
| Дополнительный recency | `days_since_last_activity` | `int32` | cutoff_date − last_activity_date | Lifetime | Давность любого последнего наблюдаемого события | Не допускается | Преимущественно classifier |
| Дополнительный recency | `days_since_last_cart` | `float32` | cutoff_date − max(event_date при to_cart > 0) | Lifetime | Давность последнего добавления в корзину | NaN, если корзины не было | Преимущественно classifier |
| Дополнительный recency | `days_since_last_category_visit` | `float32` | cutoff_date − max(event_date при cat > 0) | Lifetime | Давность последней активности в каталоге | NaN, если посещений каталога не было | Преимущественно classifier |
| Дополнительный lifetime | `active_day_rate_lifetime` | `float32` | active_days / customer_age_days | Lifetime | Доля календарных дней с наблюдаемой активностью | Не допускается | Classifier + regressor |
| Дополнительный lifetime | `category_days_total` | `int32` | Число дат с cat > 0 | Lifetime | Общая частота каталожной активности | 0 означает отсутствие каталожной активности | Classifier + regressor |
| Дополнительный lifetime | `median_purchase_gap_days` | `float32` | median(diff(event_date)) по покупочным дням | Lifetime | Типичный персональный интервал между покупками | NaN, если менее двух покупочных дней | Преимущественно classifier |
| Дни с событиями | `active_days_30d` | `int32` | Число дат, где Любое наблюдение пользователя | cutoff−29 … cutoff | Число активных дней за последние 30 дней | 0 означает отсутствие соответствующих дней | Classifier + regressor |
| Дни с событиями | `active_days_prev_30d` | `int32` | Число дат, где Любое наблюдение пользователя | cutoff−59 … cutoff−30 | Число активных дней за предыдущие 30 дней | 0 означает отсутствие соответствующих дней | Classifier + regressor |
| Дни с событиями | `active_days_90d` | `int32` | Число дат, где Любое наблюдение пользователя | cutoff−89 … cutoff | Число активных дней за последние 90 дней | 0 означает отсутствие соответствующих дней | Classifier + regressor |
| Дни с событиями | `search_days_30d` | `int32` | Число дат, где searches > 0 | cutoff−29 … cutoff | Число поисковых дней за последние 30 дней | 0 означает отсутствие соответствующих дней | Classifier + regressor |
| Дни с событиями | `search_days_prev_30d` | `int32` | Число дат, где searches > 0 | cutoff−59 … cutoff−30 | Число поисковых дней за предыдущие 30 дней | 0 означает отсутствие соответствующих дней | Classifier + regressor |
| Дни с событиями | `search_days_90d` | `int32` | Число дат, где searches > 0 | cutoff−89 … cutoff | Число поисковых дней за последние 90 дней | 0 означает отсутствие соответствующих дней | Classifier + regressor |
| Дни с событиями | `category_days_30d` | `int32` | Число дат, где cat > 0 | cutoff−29 … cutoff | Число каталожных дней за последние 30 дней | 0 означает отсутствие соответствующих дней | Classifier + regressor |
| Дни с событиями | `category_days_prev_30d` | `int32` | Число дат, где cat > 0 | cutoff−59 … cutoff−30 | Число каталожных дней за предыдущие 30 дней | 0 означает отсутствие соответствующих дней | Classifier + regressor |
| Дни с событиями | `category_days_90d` | `int32` | Число дат, где cat > 0 | cutoff−89 … cutoff | Число каталожных дней за последние 90 дней | 0 означает отсутствие соответствующих дней | Classifier + regressor |
| Дни с событиями | `cart_days_30d` | `int32` | Число дат, где to_cart > 0 | cutoff−29 … cutoff | Число корзинных дней за последние 30 дней | 0 означает отсутствие соответствующих дней | Classifier + regressor |
| Дни с событиями | `cart_days_prev_30d` | `int32` | Число дат, где to_cart > 0 | cutoff−59 … cutoff−30 | Число корзинных дней за предыдущие 30 дней | 0 означает отсутствие соответствующих дней | Classifier + regressor |
| Дни с событиями | `cart_days_90d` | `int32` | Число дат, где to_cart > 0 | cutoff−89 … cutoff | Число корзинных дней за последние 90 дней | 0 означает отсутствие соответствующих дней | Classifier + regressor |
| Дни с событиями | `purchase_days_30d` | `int32` | Число дат, где to_ord > 0 | cutoff−29 … cutoff | Число покупочных дней за последние 30 дней | 0 означает отсутствие соответствующих дней | Classifier + regressor |
| Дни с событиями | `purchase_days_prev_30d` | `int32` | Число дат, где to_ord > 0 | cutoff−59 … cutoff−30 | Число покупочных дней за предыдущие 30 дней | 0 означает отсутствие соответствующих дней | Classifier + regressor |
| Дни с событиями | `purchase_days_90d` | `int32` | Число дат, где to_ord > 0 | cutoff−89 … cutoff | Число покупочных дней за последние 90 дней | 0 означает отсутствие соответствующих дней | Classifier + regressor |
| Канальные воронки | `search_query_to_cart_rate` | `float32` | sum(search_to_cart) / searches_total | Lifetime | Добавленных через поиск товаров на один запрос | NaN, если поисков не было | Преимущественно classifier |
| Канальные воронки | `search_cart_to_purchase_rate` | `float32` | sum(search_to_ord) / sum(search_to_cart) | Lifetime | Конверсия товарных добавлений поискового канала в покупки | NaN, если поисковых добавлений не было | Classifier + regressor |
| Канальные воронки | `category_cart_to_purchase_rate` | `float32` | sum(cat_to_ord) / sum(cat_to_cart) | Lifetime | Конверсия товарных добавлений каталога в покупки | NaN, если каталожных добавлений не было | Classifier + regressor |
| Канальные воронки | `search_cart_share` | `float32` | sum(search_to_cart) / cart_items_total | Lifetime | Доля добавлений в корзину, пришедших из поиска | NaN, если добавлений не было | Classifier + regressor |
| Канальные воронки | `search_purchase_share` | `float32` | sum(search_to_ord) / purchased_items_total | Lifetime | Доля купленных товаров поискового канала | NaN, если покупок не было | Classifier + regressor |
| Канальный GMV | `gmv_search_30d` | `float32` | sum(gmv_search) | cutoff−29 … cutoff | GMV поискового канала за последние 30 дней | 0 означает отсутствие GMV | Преимущественно regressor |
| Канальный GMV | `gmv_search_90d` | `float32` | sum(gmv_search) | cutoff−89 … cutoff | GMV поискового канала за последние 90 дней | 0 означает отсутствие GMV | Преимущественно regressor |
| Канальный GMV | `search_gmv_share_30d` | `float32` | gmv_search_30d / gmv_30d | Последние 30 дней | Недавняя доля GMV поискового канала | NaN при нулевом gmv_30d | Classifier + regressor |
| Канальный GMV | `search_gmv_share_90d` | `float32` | gmv_search_90d / gmv_90d | Последние 90 дней | Среднесрочная доля GMV поискового канала | NaN при нулевом gmv_90d | Classifier + regressor |
| Денежный профиль | `gmv_per_purchase_day` | `float32` | gmv_total / purchase_days | Lifetime | Средний GMV на покупочный день | NaN, если покупок не было | Преимущественно regressor |
| Денежный профиль | `gmv_daily_std_90d` | `float32` | std(gmv по календарным дням, включая нулевые дни) | Последние доступные 90 дней | Волатильность дневного GMV | Не допускается | Преимущественно regressor |
| Дополнительные тренды | `gmv_trend_log` | `float32` | log1p(gmv_30d) − log1p(gmv_prev_30d) | 30d против prev30d | Изменение недавнего GMV | Не допускается | Преимущественно regressor |
| Дополнительные тренды | `searches_trend_log` | `float32` | log1p(searches_30d) − log1p(searches_prev_30d) | 30d против prev30d | Изменение поисковой интенсивности | Не допускается | Преимущественно classifier |
| Дополнительные тренды | `purchased_items_trend_log` | `float32` | log1p(purchased_items_30d) − log1p(purchased_items_prev_30d) | 30d против prev30d | Изменение объёма покупок | Не допускается | Classifier + regressor |
| Дополнительные тренды | `active_days_trend_log` | `float32` | log1p(active_days_30d) − log1p(active_days_prev_30d) | 30d против prev30d | Изменение регулярности активности | Не допускается | Преимущественно classifier |
| Целевые | `target_gmv_30d` | `float64` | sum(gmv) при cutoff < event_date ≤ cutoff + 30 дней | Будущие 30 дней | Суммарный GMV для регрессора и итоговой задачи | Null только для cutoff 2026-02-13 | Target регрессии |
| Целевые | `target_nonzero` | `nullable int8` | 1, если target_gmv_30d > 0, иначе 0 | Будущие 30 дней | Цель классификатора первой стадии Hurdle | Null только для cutoff 2026-02-13 | Target классификации |


## Критерии готовности

- 2 597 330 строк, 95 колонок и 11 cutoff.
- 2 347 330 размеченных строк и 250 000 inference-строк.
- Нет дубликатов `user_id × cutoff_date`, бесконечностей и временной утечки.
- Target неотрицателен, `target_nonzero` согласован с `target_gmv_30d`, а null target встречается только на cutoff `2026-02-13`.
- Типы и порядок колонок совпадают с приведённым словарём.
- Сумма target каждого размеченного cutoff независимо сверяется с исходными событиями.
- Синтетические boundary-тесты и сравнение с простой эталонной реализацией проходят до полного запуска.


## Импорты, пути и физическая схема

Используются только стандартная библиотека, NumPy и PyArrow. Путь к корню проекта определяется независимо от текущей рабочей папки. Порядок и типы колонок задаются одной Arrow-схемой.


In [1]:
from pathlib import Path
import shutil
import time

import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "train.parquet").exists() and (
            candidate / "notebooks"
        ).exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root containing data/train.parquet")


PROJECT_ROOT = find_project_root()
INPUT_PATH = PROJECT_ROOT / "data" / "train.parquet"
OUTPUT_PATH = PROJECT_ROOT / "data" / "Prepared_data.parquet"
TEMP_ROOT = PROJECT_ROOT / "tmp" / "prepared_data_snapshots"

LABELED_CUTOFF_STRINGS = [
    "2025-04-19",
    "2025-05-19",
    "2025-06-18",
    "2025-07-18",
    "2025-08-17",
    "2025-09-16",
    "2025-10-16",
    "2025-11-15",
    "2025-12-15",
    "2026-01-14",
]
INFERENCE_CUTOFF_STRING = "2026-02-13"
CUTOFF_STRINGS = [*LABELED_CUTOFF_STRINGS, INFERENCE_CUTOFF_STRING]
CUTOFF_DAYS = np.array(CUTOFF_STRINGS, dtype="datetime64[D]").astype(np.int32)
INFERENCE_CUTOFF_DAY = CUTOFF_DAYS[-1]

RAW_COLUMNS = [
    "event_date",
    "user_id",
    "searches",
    "cat",
    "search_to_cart",
    "search_to_ord",
    "cat_to_cart",
    "cat_to_ord",
    "gmv_search",
    "gmv_cat",
    "to_cart",
    "to_ord",
    "gmv",
]

SERVICE_COLUMNS = ["user_id", "cutoff_date"]
TARGET_COLUMNS = ["target_gmv_30d", "target_nonzero"]

BASE_FEATURES = [
    "searches_total",
    "cart_items_total",
    "purchased_items_total",
    "search_purchased_items_total",
    "gmv_total",
    "gmv_search_total",
    "gmv_cat_total",
    "purchase_days",
]

RFM_FEATURES = [
    "days_since_last_purchase",
    "never_purchased",
    "days_since_last_search",
    "never_searched",
    "recency_ratio",
    "active_days",
    "purchase_frequency",
    "search_to_purchase_freq",
    "gmv_per_item",
    "gmv_daily_mean",
]

FUNNEL_FEATURES = [
    "sci_items_per_query",
    "cart_abandonment_rate",
    "cart_purchase_exceeds_cart",
    "search_gmv_share",
]

WINDOW_FEATURES = [
    f"{metric}_{window}"
    for metric in ["searches", "cart_items", "purchased_items", "gmv"]
    for window in ["7d", "30d", "prev_30d", "90d"]
]

VELOCITY_FEATURES = [
    f"{metric}_velocity_{comparison}"
    for metric in ["search", "cart", "purchase"]
    for comparison in ["7d_vs_30d", "30d_vs_lifetime"]
]

ORIGINAL_TREND_FEATURES = ["intent_trend_log"]

STABILITY_FEATURES = [
    "engagement_daily_mean_90d",
    "engagement_cv_90d",
    "no_recent_engagement",
]

AGE_FEATURES = ["customer_age_days", "customer_age_left_censored"]

SEASONALITY_FEATURES = [
    "last_visit_weekday_sin",
    "last_visit_weekday_cos",
    "cutoff_month_sin",
    "cutoff_month_cos",
]

WHALE_FEATURES = ["whale_score"]

ADDITIONAL_RECENCY_FEATURES = [
    "days_since_last_activity",
    "days_since_last_cart",
    "days_since_last_category_visit",
    "active_day_rate_lifetime",
    "category_days_total",
    "median_purchase_gap_days",
]

EVENT_DAY_FEATURES = [
    f"{event}_days_{window}"
    for event in ["active", "search", "category", "cart", "purchase"]
    for window in ["30d", "prev_30d", "90d"]
]

CHANNEL_FUNNEL_FEATURES = [
    "search_query_to_cart_rate",
    "search_cart_to_purchase_rate",
    "category_cart_to_purchase_rate",
    "search_cart_share",
    "search_purchase_share",
]

CHANNEL_GMV_FEATURES = [
    "gmv_search_30d",
    "gmv_search_90d",
    "search_gmv_share_30d",
    "search_gmv_share_90d",
]

MONETARY_PROFILE_FEATURES = ["gmv_per_purchase_day", "gmv_daily_std_90d"]

ADDITIONAL_TREND_FEATURES = [
    "gmv_trend_log",
    "searches_trend_log",
    "purchased_items_trend_log",
    "active_days_trend_log",
]

FEATURE_COLUMNS = [
    *BASE_FEATURES,
    *RFM_FEATURES,
    *FUNNEL_FEATURES,
    *WINDOW_FEATURES,
    *VELOCITY_FEATURES,
    *ORIGINAL_TREND_FEATURES,
    *STABILITY_FEATURES,
    *AGE_FEATURES,
    *SEASONALITY_FEATURES,
    *WHALE_FEATURES,
    *ADDITIONAL_RECENCY_FEATURES,
    *EVENT_DAY_FEATURES,
    *CHANNEL_FUNNEL_FEATURES,
    *CHANNEL_GMV_FEATURES,
    *MONETARY_PROFILE_FEATURES,
    *ADDITIONAL_TREND_FEATURES,
]

INT8_FEATURES = {
    "never_purchased",
    "never_searched",
    "cart_purchase_exceeds_cart",
    "no_recent_engagement",
    "customer_age_left_censored",
}

INT32_FEATURES = {
    "searches_total",
    "cart_items_total",
    "purchased_items_total",
    "search_purchased_items_total",
    "purchase_days",
    "active_days",
    "customer_age_days",
    "days_since_last_activity",
    "category_days_total",
    *[name for name in WINDOW_FEATURES if not name.startswith("gmv_")],
    *EVENT_DAY_FEATURES,
}


def arrow_type(name):
    if name == "user_id":
        return pa.int64()
    if name == "cutoff_date":
        return pa.date32()
    if name in INT8_FEATURES or name == "target_nonzero":
        return pa.int8()
    if name in INT32_FEATURES:
        return pa.int32()
    if name == "target_gmv_30d":
        return pa.float64()
    return pa.float32()


ALL_COLUMNS = [*SERVICE_COLUMNS, *FEATURE_COLUMNS, *TARGET_COLUMNS]
COLUMN_SPECS = [{"name": name, "type": arrow_type(name)} for name in ALL_COLUMNS]
OUTPUT_SCHEMA = pa.schema([pa.field(item["name"], item["type"]) for item in COLUMN_SPECS])
SHARD_COLUMNS = [name for name in ALL_COLUMNS if name != "whale_score"]
SHARD_SCHEMA = pa.schema([OUTPUT_SCHEMA.field(name) for name in SHARD_COLUMNS])

assert len(FEATURE_COLUMNS) == 91
assert len(ALL_COLUMNS) == 95
assert len(set(ALL_COLUMNS)) == 95


## Расчёт snapshot одного пользователя

История пользователя уже отсортирована по дате. Кумулятивные суммы позволяют получать lifetime- и оконные агрегаты за O(1), а `searchsorted` фиксирует точные границы history и target без просмотра будущих событий.


In [2]:
def _prefix(values, dtype=np.float64):
    return np.concatenate(([0], np.cumsum(values, dtype=dtype)))


def _interval_sum(prefix_values, left, right):
    return prefix_values[right] - prefix_values[left]


def _safe_ratio(numerator, denominator):
    if denominator == 0 or np.isnan(denominator):
        return np.nan
    return numerator / denominator


def _last_date_before(event_days, history_end_day):
    count = np.searchsorted(event_days, history_end_day, side="right")
    return None if count == 0 else int(event_days[count - 1])


def _month_zero(day_number):
    date_value = np.datetime64("1970-01-01", "D") + np.timedelta64(int(day_number), "D")
    return int(date_value.astype("datetime64[M]").astype(np.int64) % 12)


def compute_user_snapshots(columns, dataset_start_day):
    dates = np.asarray(columns["event_date"], dtype=np.int32)
    if len(dates) == 0:
        return {name: [] for name in SHARD_COLUMNS}
    if np.any(dates[1:] <= dates[:-1]):
        raise ValueError("Dates must be strictly increasing within each user")

    user_id = int(columns["user_id"][0])
    arrays = {
        name: np.asarray(columns[name])
        for name in RAW_COLUMNS
        if name not in {"event_date", "user_id"}
    }
    if any(len(values) != len(dates) for values in arrays.values()):
        raise ValueError("All per-user columns must have equal lengths")

    searches = arrays["searches"]
    category = arrays["cat"]
    to_cart = arrays["to_cart"]
    to_ord = arrays["to_ord"]
    gmv = arrays["gmv"].astype(np.float64, copy=False)

    indicators = {
        "active": np.ones(len(dates), dtype=np.int8),
        "search": searches > 0,
        "category": category > 0,
        "cart": to_cart > 0,
        "purchase": to_ord > 0,
    }
    additive_names = [
        "searches",
        "search_to_cart",
        "search_to_ord",
        "cat_to_cart",
        "cat_to_ord",
        "to_cart",
        "to_ord",
        "gmv_search",
        "gmv_cat",
        "gmv",
    ]
    prefixes = {name: _prefix(arrays[name]) for name in additive_names}
    indicator_prefixes = {name: _prefix(values, dtype=np.int64) for name, values in indicators.items()}

    engagement = searches.astype(np.float64) + to_cart.astype(np.float64) + to_ord.astype(np.float64)
    engagement_prefix = _prefix(engagement)
    engagement_sq_prefix = _prefix(engagement * engagement)
    gmv_sq_prefix = _prefix(gmv * gmv)

    event_dates = {
        name: dates[mask]
        for name, mask in indicators.items()
        if name != "active"
    }

    output = {name: [] for name in SHARD_COLUMNS}
    first_activity = int(dates[0])

    for cutoff in CUTOFF_DAYS:
        cutoff = int(cutoff)
        history_end = int(np.searchsorted(dates, cutoff, side="right"))
        if history_end == 0:
            continue

        left_7 = int(np.searchsorted(dates, cutoff - 6, side="left"))
        left_30 = int(np.searchsorted(dates, cutoff - 29, side="left"))
        left_prev_30 = int(np.searchsorted(dates, cutoff - 59, side="left"))
        right_prev_30 = int(np.searchsorted(dates, cutoff - 29, side="left"))
        left_90 = int(np.searchsorted(dates, cutoff - 89, side="left"))

        bounds = {
            "7d": (left_7, history_end),
            "30d": (left_30, history_end),
            "prev_30d": (left_prev_30, right_prev_30),
            "90d": (left_90, history_end),
        }

        customer_age = cutoff - first_activity + 1
        exposure_7 = min(customer_age, 7)
        exposure_30 = min(customer_age, 30)
        exposure_90 = min(customer_age, 90)

        totals = {name: _interval_sum(prefixes[name], 0, history_end) for name in additive_names}
        day_totals = {
            name: int(_interval_sum(prefix_values, 0, history_end))
            for name, prefix_values in indicator_prefixes.items()
        }
        window_values = {
            name: {
                suffix: _interval_sum(prefixes[name], left, right)
                for suffix, (left, right) in bounds.items()
            }
            for name in ["searches", "to_cart", "to_ord", "gmv", "gmv_search"]
        }
        window_days = {
            name: {
                suffix: int(_interval_sum(indicator_prefixes[name], left, right))
                for suffix, (left, right) in bounds.items()
            }
            for name in indicators
        }

        last_purchase = _last_date_before(event_dates["purchase"], cutoff)
        last_search = _last_date_before(event_dates["search"], cutoff)
        last_cart = _last_date_before(event_dates["cart"], cutoff)
        last_category = _last_date_before(event_dates["category"], cutoff)

        purchase_date_count = np.searchsorted(event_dates["purchase"], cutoff, side="right")
        if purchase_date_count >= 2:
            median_purchase_gap = float(
                np.median(np.diff(event_dates["purchase"][:purchase_date_count]))
            )
        else:
            median_purchase_gap = np.nan

        days_since_purchase = np.nan if last_purchase is None else cutoff - last_purchase
        days_since_search = np.nan if last_search is None else cutoff - last_search
        cart_rate = _safe_ratio(totals["to_ord"], totals["to_cart"])

        recent_engagement_sum = _interval_sum(engagement_prefix, left_90, history_end)
        recent_engagement_sq_sum = _interval_sum(engagement_sq_prefix, left_90, history_end)
        engagement_mean = recent_engagement_sum / exposure_90
        engagement_variance = max(
            recent_engagement_sq_sum / exposure_90 - engagement_mean * engagement_mean,
            0.0,
        )
        engagement_std = np.sqrt(engagement_variance)

        recent_gmv_sum = window_values["gmv"]["90d"]
        recent_gmv_sq_sum = _interval_sum(gmv_sq_prefix, left_90, history_end)
        recent_gmv_mean = recent_gmv_sum / exposure_90
        recent_gmv_variance = max(
            recent_gmv_sq_sum / exposure_90 - recent_gmv_mean * recent_gmv_mean,
            0.0,
        )

        last_activity = int(dates[history_end - 1])
        weekday = (last_activity + 3) % 7
        cutoff_month_zero = _month_zero(cutoff)

        row = {
            "user_id": user_id,
            "cutoff_date": cutoff,
            "searches_total": int(totals["searches"]),
            "cart_items_total": int(totals["to_cart"]),
            "purchased_items_total": int(totals["to_ord"]),
            "search_purchased_items_total": int(totals["search_to_ord"]),
            "gmv_total": totals["gmv"],
            "gmv_search_total": totals["gmv_search"],
            "gmv_cat_total": totals["gmv_cat"],
            "purchase_days": day_totals["purchase"],
            "days_since_last_purchase": days_since_purchase,
            "never_purchased": int(last_purchase is None),
            "days_since_last_search": days_since_search,
            "never_searched": int(last_search is None),
            "recency_ratio": _safe_ratio(days_since_purchase, median_purchase_gap),
            "active_days": day_totals["active"],
            "purchase_frequency": _safe_ratio(day_totals["purchase"], day_totals["active"]),
            "search_to_purchase_freq": _safe_ratio(totals["searches"], totals["to_ord"]),
            "gmv_per_item": _safe_ratio(totals["gmv"], totals["to_ord"]),
            "gmv_daily_mean": totals["gmv"] / customer_age,
            "sci_items_per_query": _safe_ratio(totals["search_to_ord"], totals["searches"]),
            "cart_abandonment_rate": np.nan if np.isnan(cart_rate) else float(np.clip(1.0 - cart_rate, 0.0, 1.0)),
            "cart_purchase_exceeds_cart": int(not np.isnan(cart_rate) and cart_rate > 1.0),
            "search_gmv_share": _safe_ratio(totals["gmv_search"], totals["gmv"]),
            "intent_trend_log": np.log1p(window_values["to_cart"]["30d"]) - np.log1p(window_values["to_cart"]["prev_30d"]),
            "engagement_daily_mean_90d": engagement_mean,
            "engagement_cv_90d": _safe_ratio(engagement_std, engagement_mean),
            "no_recent_engagement": int(recent_engagement_sum == 0),
            "customer_age_days": customer_age,
            "customer_age_left_censored": int(first_activity == int(dataset_start_day)),
            "last_visit_weekday_sin": np.sin(2.0 * np.pi * weekday / 7.0),
            "last_visit_weekday_cos": np.cos(2.0 * np.pi * weekday / 7.0),
            "cutoff_month_sin": np.sin(2.0 * np.pi * cutoff_month_zero / 12.0),
            "cutoff_month_cos": np.cos(2.0 * np.pi * cutoff_month_zero / 12.0),
            "days_since_last_activity": cutoff - last_activity,
            "days_since_last_cart": np.nan if last_cart is None else cutoff - last_cart,
            "days_since_last_category_visit": np.nan if last_category is None else cutoff - last_category,
            "active_day_rate_lifetime": day_totals["active"] / customer_age,
            "category_days_total": day_totals["category"],
            "median_purchase_gap_days": median_purchase_gap,
            "search_query_to_cart_rate": _safe_ratio(totals["search_to_cart"], totals["searches"]),
            "search_cart_to_purchase_rate": _safe_ratio(totals["search_to_ord"], totals["search_to_cart"]),
            "category_cart_to_purchase_rate": _safe_ratio(totals["cat_to_ord"], totals["cat_to_cart"]),
            "search_cart_share": _safe_ratio(totals["search_to_cart"], totals["to_cart"]),
            "search_purchase_share": _safe_ratio(totals["search_to_ord"], totals["to_ord"]),
            "gmv_search_30d": window_values["gmv_search"]["30d"],
            "gmv_search_90d": window_values["gmv_search"]["90d"],
            "search_gmv_share_30d": _safe_ratio(window_values["gmv_search"]["30d"], window_values["gmv"]["30d"]),
            "search_gmv_share_90d": _safe_ratio(window_values["gmv_search"]["90d"], window_values["gmv"]["90d"]),
            "gmv_per_purchase_day": _safe_ratio(totals["gmv"], day_totals["purchase"]),
            "gmv_daily_std_90d": np.sqrt(recent_gmv_variance),
            "gmv_trend_log": np.log1p(window_values["gmv"]["30d"]) - np.log1p(window_values["gmv"]["prev_30d"]),
            "searches_trend_log": np.log1p(window_values["searches"]["30d"]) - np.log1p(window_values["searches"]["prev_30d"]),
            "purchased_items_trend_log": np.log1p(window_values["to_ord"]["30d"]) - np.log1p(window_values["to_ord"]["prev_30d"]),
            "active_days_trend_log": np.log1p(window_days["active"]["30d"]) - np.log1p(window_days["active"]["prev_30d"]),
        }

        for raw_name, feature_name in [
            ("searches", "searches"),
            ("to_cart", "cart_items"),
            ("to_ord", "purchased_items"),
            ("gmv", "gmv"),
        ]:
            for suffix in ["7d", "30d", "prev_30d", "90d"]:
                value = window_values[raw_name][suffix]
                row[f"{feature_name}_{suffix}"] = value if raw_name == "gmv" else int(value)

        for metric_name, source_name, total_name in [
            ("search", "searches", "searches"),
            ("cart", "to_cart", "to_cart"),
            ("purchase", "to_ord", "to_ord"),
        ]:
            rate_7 = window_values[source_name]["7d"] / exposure_7
            rate_30 = window_values[source_name]["30d"] / exposure_30
            lifetime_rate = totals[total_name] / customer_age
            row[f"{metric_name}_velocity_7d_vs_30d"] = _safe_ratio(rate_7, rate_30)
            row[f"{metric_name}_velocity_30d_vs_lifetime"] = _safe_ratio(rate_30, lifetime_rate)

        for event_name in ["active", "search", "category", "cart", "purchase"]:
            for suffix in ["30d", "prev_30d", "90d"]:
                row[f"{event_name}_days_{suffix}"] = window_days[event_name][suffix]

        if cutoff == int(INFERENCE_CUTOFF_DAY):
            row["target_gmv_30d"] = None
            row["target_nonzero"] = None
        else:
            target_end = int(np.searchsorted(dates, cutoff + 30, side="right"))
            target_gmv = float(_interval_sum(prefixes["gmv"], history_end, target_end))
            row["target_gmv_30d"] = target_gmv
            row["target_nonzero"] = int(target_gmv > 0)

        if set(row) != set(SHARD_COLUMNS):
            missing = sorted(set(SHARD_COLUMNS) - set(row))
            extra = sorted(set(row) - set(SHARD_COLUMNS))
            raise AssertionError(f"Snapshot schema mismatch; missing={missing}, extra={extra}")
        for name in SHARD_COLUMNS:
            output[name].append(row[name])

    return output


## Потоковое чтение и временные snapshot

PyArrow читает один row group за раз. Если история последнего пользователя продолжается в следующем блоке, она переносится целиком. Готовые строки записываются в 11 временных Parquet-shard, не накапливая весь датасет в RAM.


In [3]:
def iter_complete_user_blocks(parquet_file):
    carry = None
    for row_group_index in range(parquet_file.num_row_groups):
        table = parquet_file.read_row_group(row_group_index, columns=RAW_COLUMNS)
        if carry is not None:
            table = pa.concat_tables([carry, table])

        users = table["user_id"].combine_chunks().to_numpy(zero_copy_only=False)
        if np.any(users[1:] < users[:-1]):
            raise ValueError("Input Parquet must be sorted by user_id")

        last_user = users[-1]
        split = int(np.searchsorted(users, last_user, side="left"))
        complete = table.slice(0, split)
        carry = table.slice(split)
        if complete.num_rows:
            yield complete

    if carry is not None and carry.num_rows:
        yield carry


def _table_from_columns(columns, schema):
    arrays = []
    for field in schema:
        values = columns[field.name]
        if pa.types.is_date32(field.type):
            array = pa.array(values, type=pa.int32()).cast(pa.date32())
        else:
            array = pa.array(values, type=field.type)
        arrays.append(array)
    return pa.Table.from_arrays(arrays, schema=schema)


def write_snapshot_shards(input_path, temp_root, dataset_start_day, progress=False):
    input_path = Path(input_path)
    temp_root = Path(temp_root)
    temp_root.mkdir(parents=True, exist_ok=False)

    shard_paths = {
        int(cutoff): temp_root / f"snapshot_{date_string}.parquet"
        for cutoff, date_string in zip(CUTOFF_DAYS, CUTOFF_STRINGS)
    }
    writers = {
        cutoff: pq.ParquetWriter(
            path,
            SHARD_SCHEMA,
            compression="zstd",
            compression_level=3,
            use_dictionary=False,
        )
        for cutoff, path in shard_paths.items()
    }

    try:
        parquet_file = pq.ParquetFile(input_path, memory_map=True)
        started = time.perf_counter()
        for block_index, block in enumerate(iter_complete_user_blocks(parquet_file), start=1):
            block_columns = {
                name: (
                    block[name]
                    .combine_chunks()
                    .cast(pa.int32())
                    .to_numpy(zero_copy_only=False)
                    if name == "event_date"
                    else block[name].combine_chunks().to_numpy(zero_copy_only=False)
                )
                for name in RAW_COLUMNS
            }
            users = block_columns["user_id"]
            starts = np.r_[0, np.flatnonzero(users[1:] != users[:-1]) + 1]
            ends = np.r_[starts[1:], len(users)]
            prepared = {name: [] for name in SHARD_COLUMNS}

            for start, end in zip(starts, ends):
                user_columns = {
                    name: values[start:end]
                    for name, values in block_columns.items()
                }
                user_rows = compute_user_snapshots(user_columns, dataset_start_day)
                for name in SHARD_COLUMNS:
                    prepared[name].extend(user_rows[name])

            if not prepared["user_id"]:
                continue
            prepared_table = _table_from_columns(prepared, SHARD_SCHEMA)
            for cutoff, writer in writers.items():
                cutoff_scalar = pa.scalar(cutoff, type=pa.int32()).cast(pa.date32())
                snapshot = prepared_table.filter(
                    pc.equal(prepared_table["cutoff_date"], cutoff_scalar)
                )
                if snapshot.num_rows:
                    writer.write_table(snapshot, row_group_size=65_536)
            if progress and (
                block_index == 1
                or block_index % 25 == 0
                or block_index == parquet_file.num_row_groups
            ):
                elapsed = time.perf_counter() - started
                print(
                    f"Processed source block {block_index}/{parquet_file.num_row_groups} "
                    f"in {elapsed:,.1f}s"
                )
    finally:
        for writer in writers.values():
            writer.close()

    return shard_paths


def inspect_source(path):
    parquet_file = pq.ParquetFile(path, memory_map=True)
    schema = parquet_file.schema_arrow
    missing = sorted(set(RAW_COLUMNS) - set(schema.names))
    if missing:
        raise KeyError(f"Input Parquet is missing columns: {missing}")

    event_index = schema.get_field_index("event_date")
    minimums = []
    maximums = []
    nulls = 0
    for row_group_index in range(parquet_file.num_row_groups):
        statistics = parquet_file.metadata.row_group(row_group_index).column(
            event_index
        ).statistics
        if statistics is None or not statistics.has_min_max:
            raise ValueError("event_date statistics are required in every row group")
        minimums.append(statistics.min)
        maximums.append(statistics.max)
        nulls += statistics.null_count or 0

    start = np.datetime64(min(minimums), "D").astype(np.int32).item()
    end = np.datetime64(max(maximums), "D").astype(np.int32).item()
    return {
        "rows": parquet_file.metadata.num_rows,
        "row_groups": parquet_file.num_row_groups,
        "columns": parquet_file.metadata.num_columns,
        "dataset_start_day": start,
        "dataset_end_day": end,
        "event_date_nulls": nulls,
    }


def raw_target_totals(input_path):
    totals = {int(cutoff): 0.0 for cutoff in CUTOFF_DAYS[:-1]}
    parquet_file = pq.ParquetFile(input_path, memory_map=True)
    for block in iter_complete_user_blocks(parquet_file):
        users = block["user_id"].combine_chunks().to_numpy(zero_copy_only=False)
        dates = block["event_date"].combine_chunks().cast(pa.int32()).to_numpy(
            zero_copy_only=False
        )
        gmv = block["gmv"].combine_chunks().to_numpy(zero_copy_only=False)
        starts = np.r_[0, np.flatnonzero(users[1:] != users[:-1]) + 1]
        ends = np.r_[starts[1:], len(users)]
        for start, end in zip(starts, ends):
            user_dates = dates[start:end]
            user_gmv = gmv[start:end]
            first_activity = int(user_dates[0])
            for cutoff in CUTOFF_DAYS[:-1]:
                cutoff = int(cutoff)
                if first_activity > cutoff:
                    continue
                left = int(np.searchsorted(user_dates, cutoff, side="right"))
                right = int(np.searchsorted(user_dates, cutoff + 30, side="right"))
                totals[cutoff] += float(user_gmv[left:right].sum(dtype=np.float64))
    return totals


## Whale-rank, консолидация и проверки

После появления полной когорты cutoff рассчитывается percentile-rank `gmv_90d`. Shard объединяются в один файл, отсортированный по `cutoff_date, user_id`. До атомарной замены проверяются схема, target, дубликаты, сортировка, диапазоны и бесконечности.


In [4]:
def average_percentile_rank(values):
    values = np.asarray(values, dtype=np.float64)
    if values.ndim != 1 or len(values) == 0:
        raise ValueError("values must be a non-empty one-dimensional array")
    order = np.argsort(values, kind="mergesort")
    sorted_values = values[order]
    starts = np.r_[0, np.flatnonzero(sorted_values[1:] != sorted_values[:-1]) + 1]
    ends = np.r_[starts[1:], len(values)]
    average_ranks = ((starts + 1 + ends) / 2.0) / len(values)
    ranked_sorted = np.repeat(average_ranks, ends - starts)
    result = np.empty(len(values), dtype=np.float32)
    result[order] = ranked_sorted.astype(np.float32)
    return result


def consolidate_shards(shard_paths, output_path):
    output_path = Path(output_path)
    temporary_output = output_path.with_name(output_path.name + ".tmp")
    if temporary_output.exists():
        temporary_output.unlink()

    whale_index = ALL_COLUMNS.index("whale_score")
    writer = pq.ParquetWriter(
        temporary_output,
        OUTPUT_SCHEMA,
        compression="zstd",
        compression_level=3,
        use_dictionary=["cutoff_date"],
    )
    try:
        for cutoff in CUTOFF_DAYS:
            table = pq.read_table(shard_paths[int(cutoff)])
            ranks = average_percentile_rank(
                table["gmv_90d"].combine_chunks().to_numpy(zero_copy_only=False)
            )
            table = table.add_column(
                whale_index,
                pa.field("whale_score", pa.float32()),
                pa.array(ranks, type=pa.float32()),
            )
            table = table.select(ALL_COLUMNS).cast(OUTPUT_SCHEMA)
            writer.write_table(table, row_group_size=65_536)
    finally:
        writer.close()

    verify_prepared_data(temporary_output)
    temporary_output.replace(output_path)
    return output_path


def verify_prepared_data(path, expected_rows=None, expected_inference_rows=None):
    parquet_file = pq.ParquetFile(path, memory_map=True)
    if not parquet_file.schema_arrow.equals(OUTPUT_SCHEMA, check_metadata=False):
        raise AssertionError("Prepared Parquet schema does not match OUTPUT_SCHEMA")

    rows = parquet_file.metadata.num_rows
    duplicate_keys = 0
    order_violations = 0
    infinite_values = 0
    inference_rows = 0
    cutoff_values = set()
    target_totals = {int(cutoff): 0.0 for cutoff in CUTOFF_DAYS}
    previous_cutoff = None
    previous_user = None

    for batch in parquet_file.iter_batches(batch_size=65_536, columns=ALL_COLUMNS):
        users = batch.column("user_id").to_numpy(zero_copy_only=False)
        cutoffs = (
            batch.column("cutoff_date")
            .cast(pa.int32())
            .to_numpy(zero_copy_only=False)
        )
        target_array = batch.column("target_gmv_30d")
        target_null = target_array.is_null().to_numpy(zero_copy_only=False)
        targets = target_array.to_numpy(zero_copy_only=False)
        flags = pc.fill_null(batch.column("target_nonzero"), -1).to_numpy(
            zero_copy_only=False
        )
        inference_mask = cutoffs == int(INFERENCE_CUTOFF_DAY)

        if not np.array_equal(target_null, inference_mask):
            raise AssertionError("Target nulls must occur only on the inference cutoff")
        if np.any(~inference_mask & (targets < 0)):
            raise AssertionError("Labeled target_gmv_30d contains negative values")
        if np.any(~inference_mask & (flags != (targets > 0).astype(np.int8))):
            raise AssertionError("target_nonzero disagrees with target_gmv_30d")

        inference_rows += int(np.count_nonzero(inference_mask))
        cutoff_values.update(int(value) for value in np.unique(cutoffs))
        for cutoff in np.unique(cutoffs[~inference_mask]):
            target_totals[int(cutoff)] += float(
                np.nansum(targets[cutoffs == cutoff], dtype=np.float64)
            )

        if previous_cutoff is not None:
            if cutoffs[0] < previous_cutoff or (
                cutoffs[0] == previous_cutoff and users[0] < previous_user
            ):
                order_violations += 1
            if cutoffs[0] == previous_cutoff and users[0] == previous_user:
                duplicate_keys += 1

        same_cutoff = cutoffs[1:] == cutoffs[:-1]
        duplicate_keys += int(
            np.count_nonzero(same_cutoff & (users[1:] == users[:-1]))
        )
        order_violations += int(
            np.count_nonzero(
                (cutoffs[1:] < cutoffs[:-1])
                | (same_cutoff & (users[1:] < users[:-1]))
            )
        )
        previous_cutoff = int(cutoffs[-1])
        previous_user = int(users[-1])

        for name in FEATURE_COLUMNS:
            field = OUTPUT_SCHEMA.field(name)
            if pa.types.is_floating(field.type):
                values = batch.column(name).to_numpy(zero_copy_only=False)
                infinite_values += int(np.count_nonzero(np.isinf(values)))

        whale = batch.column("whale_score").to_numpy(zero_copy_only=False)
        abandonment = batch.column("cart_abandonment_rate").to_numpy(
            zero_copy_only=False
        )
        if np.any((whale < 0) | (whale > 1)):
            raise AssertionError("whale_score is outside [0, 1]")
        finite_abandonment = abandonment[~np.isnan(abandonment)]
        if np.any((finite_abandonment < 0) | (finite_abandonment > 1)):
            raise AssertionError("cart_abandonment_rate is outside [0, 1]")

    if duplicate_keys:
        raise AssertionError(f"Found {duplicate_keys} duplicate user-cutoff keys")
    if order_violations:
        raise AssertionError(f"Found {order_violations} output ordering violations")
    if infinite_values:
        raise AssertionError(f"Found {infinite_values} infinite feature values")
    if len(cutoff_values) != len(CUTOFF_DAYS):
        raise AssertionError(f"Expected {len(CUTOFF_DAYS)} cutoffs, got {len(cutoff_values)}")
    if expected_rows is not None and rows != expected_rows:
        raise AssertionError(f"Expected {expected_rows} rows, got {rows}")
    if expected_inference_rows is not None and inference_rows != expected_inference_rows:
        raise AssertionError(
            f"Expected {expected_inference_rows} inference rows, got {inference_rows}"
        )

    return {
        "rows": rows,
        "columns": len(OUTPUT_SCHEMA),
        "cutoffs": len(cutoff_values),
        "inference_rows": inference_rows,
        "labeled_rows": rows - inference_rows,
        "duplicate_keys": duplicate_keys,
        "order_violations": order_violations,
        "infinite_values": infinite_values,
        "target_totals": target_totals,
    }


def _remove_owned_temp_directory(path):
    path = Path(path)
    if path.name not in {"prepared_data_snapshots", "shards"}:
        raise ValueError(f"Refusing to remove unrecognized temporary directory: {path}")
    if path.exists():
        shutil.rmtree(path)


def build_prepared_dataset(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    temp_root=TEMP_ROOT,
    dataset_start_day=None,
    expected_rows=2_597_330,
    expected_inference_rows=250_000,
    progress=False,
):
    input_path = Path(input_path)
    output_path = Path(output_path)
    temp_root = Path(temp_root)
    source = inspect_source(input_path)
    if source["event_date_nulls"]:
        raise ValueError("event_date contains null values")
    if dataset_start_day is None:
        dataset_start_day = source["dataset_start_day"]

    _remove_owned_temp_directory(temp_root)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    shard_paths = write_snapshot_shards(
        input_path,
        temp_root,
        dataset_start_day,
        progress=progress,
    )
    consolidate_shards(shard_paths, output_path)
    summary = verify_prepared_data(
        output_path,
        expected_rows=expected_rows,
        expected_inference_rows=expected_inference_rows,
    )

    independent_totals = raw_target_totals(input_path)
    for cutoff, expected_total in independent_totals.items():
        actual_total = summary["target_totals"][cutoff]
        if not np.isclose(actual_total, expected_total, rtol=1e-10, atol=1e-5):
            raise AssertionError(
                f"Target total mismatch for cutoff {cutoff}: "
                f"prepared={actual_total}, raw={expected_total}"
            )

    _remove_owned_temp_directory(temp_root)
    summary["source"] = source
    return summary


## Синтетические проверки границ

Литеральный пример проверяет включение дней `−89`, `−59`, `−30`, `−29`, `−6`, `0`, `+1`, `+30` и исключение `−90`, `+31`, а также средний percentile-rank при ties.


In [5]:
from tempfile import TemporaryDirectory


def _day(value):
    return np.datetime64(value, "D").astype(np.int32).item()


def _synthetic_columns(dates, gmv):
    size = len(dates)
    ones = np.ones(size, dtype=np.int64)
    zeros = np.zeros(size, dtype=np.int64)
    return {
        "event_date": np.asarray(dates, dtype=np.int32),
        "user_id": np.full(size, 7, dtype=np.int64),
        "searches": ones,
        "cat": ones,
        "search_to_cart": ones,
        "search_to_ord": ones,
        "cat_to_cart": zeros,
        "cat_to_ord": zeros,
        "gmv_search": np.asarray(gmv, dtype=np.float64),
        "gmv_cat": np.zeros(size, dtype=np.float64),
        "to_cart": ones,
        "to_ord": ones,
        "gmv": np.asarray(gmv, dtype=np.float64),
    }


cutoff = _day("2025-04-19")
dates = np.array(
    [
        cutoff - 90,
        cutoff - 89,
        cutoff - 59,
        cutoff - 30,
        cutoff - 29,
        cutoff - 7,
        cutoff - 6,
        cutoff,
        cutoff + 1,
        cutoff + 30,
        cutoff + 31,
    ],
    dtype=np.int32,
)
result = compute_user_snapshots(
    _synthetic_columns(dates, np.arange(11, dtype=np.float64)),
    dataset_start_day=cutoff - 90,
)
row_index = result["cutoff_date"].index(cutoff)
assert result["gmv_7d"][row_index] == 13.0
assert result["gmv_30d"][row_index] == 22.0
assert result["gmv_prev_30d"][row_index] == 5.0
assert result["gmv_90d"][row_index] == 28.0
assert result["target_gmv_30d"][row_index] == 17.0

np.testing.assert_allclose(
    average_percentile_rank(np.array([0.0, 0.0, 10.0, 20.0])),
    [0.375, 0.375, 0.75, 1.0],
)

print("Synthetic boundary and tied-rank checks passed.")


Synthetic boundary and tied-rank checks passed.


## Проверка исходного Parquet

До долгого запуска сверяются размер, row groups, число колонок, временной диапазон и отсутствие null в дате.


In [6]:
source_info = inspect_source(INPUT_PATH)
expected_source = {
    "rows": 30_631_006,
    "row_groups": 250,
    "columns": 18,
    "dataset_start_day": _day("2025-01-01"),
    "dataset_end_day": _day("2026-02-13"),
    "event_date_nulls": 0,
}
assert source_info == expected_source, (source_info, expected_source)
source_info


{'rows': 30631006,
 'row_groups': 250,
 'columns': 18,
 'dataset_start_day': 20089,
 'dataset_end_day': 20497,
 'event_date_nulls': 0}

## Полная сборка `Prepared_data.parquet`

Ячейка выполняет единственный потоковый проход по 30,6 млн строк, консолидирует snapshot и независимо сверяет суммы target с сырыми событиями.


In [7]:
if OUTPUT_PATH.exists():
    print("Existing Prepared_data.parquet found; verifying before reuse.")
    build_summary = verify_prepared_data(
        OUTPUT_PATH,
        expected_rows=2_597_330,
        expected_inference_rows=250_000,
    )
    independent_totals = raw_target_totals(INPUT_PATH)
    for cutoff, raw_total in independent_totals.items():
        prepared_total = build_summary["target_totals"][cutoff]
        assert np.isclose(
            prepared_total,
            raw_total,
            rtol=1e-10,
            atol=1e-5,
        ), (cutoff, prepared_total, raw_total)
    _remove_owned_temp_directory(TEMP_ROOT)
    build_seconds = None
    print("Existing dataset passed all checks and was reused.")
else:
    started = time.perf_counter()
    build_summary = build_prepared_dataset(
        input_path=INPUT_PATH,
        output_path=OUTPUT_PATH,
        temp_root=TEMP_ROOT,
        expected_rows=2_597_330,
        expected_inference_rows=250_000,
        progress=True,
    )
    build_seconds = time.perf_counter() - started
    print(f"Prepared dataset created in {build_seconds / 60:,.2f} minutes")


Existing Prepared_data.parquet found; verifying before reuse.


Existing dataset passed all checks and was reused.


## Финальный QA-отчёт

Итоговый файл открывается заново и проверяется независимо от состояния памяти после сборки.


In [8]:
final_summary = verify_prepared_data(
    OUTPUT_PATH,
    expected_rows=2_597_330,
    expected_inference_rows=250_000,
)
qa_report = {
    "path": str(OUTPUT_PATH),
    "file_size_mb": round(OUTPUT_PATH.stat().st_size / 2**20, 2),
    "rows": final_summary["rows"],
    "columns": final_summary["columns"],
    "model_features": len(FEATURE_COLUMNS),
    "cutoffs": final_summary["cutoffs"],
    "labeled_rows": final_summary["labeled_rows"],
    "inference_rows": final_summary["inference_rows"],
    "duplicate_keys": final_summary["duplicate_keys"],
    "infinite_values": final_summary["infinite_values"],
    "build_minutes": None if build_seconds is None else round(build_seconds / 60, 2),
}
qa_report


{'path': 'C:\\Users\\savin\\OneDrive\\Документы\\Мои документы\\Проекты\\E-Cup 2026\\E-CUP-2026\\data\\Prepared_data.parquet',
 'file_size_mb': 347.88,
 'rows': 2597330,
 'columns': 95,
 'model_features': 91,
 'cutoffs': 11,
 'labeled_rows': 2347330,
 'inference_rows': 250000,
 'duplicate_keys': 0,
 'infinite_values': 0,
 'build_minutes': None}

## Пример временного разделения

Исходная `cutoff_date` используется только для выбора временного среза. Она не передаётся модели как обычный признак. Благодаря физической сортировке по cutoff PyArrow пропускает ненужные row groups.


In [9]:
from datetime import date
import pyarrow.dataset as ds

prepared = ds.dataset(OUTPUT_PATH, format="parquet")
split_counts = {
    "train_through_2025-11-15": prepared.count_rows(
        filter=ds.field("cutoff_date") <= date(2025, 11, 15)
    ),
    "validation_2025-12-15": prepared.count_rows(
        filter=ds.field("cutoff_date") == date(2025, 12, 15)
    ),
    "holdout_2026-01-14": prepared.count_rows(
        filter=ds.field("cutoff_date") == date(2026, 1, 14)
    ),
    "inference_2026-02-13": prepared.count_rows(
        filter=ds.field("cutoff_date") == date(2026, 2, 13)
    ),
}
split_counts


{'train_through_2025-11-15': 1847330,
 'validation_2025-12-15': 250000,
 'holdout_2026-01-14': 250000,
 'inference_2026-02-13': 250000}